In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import MinMaxScaler
from feature_engine.discretisation import DecisionTreeDiscretiser
import xgboost as xgb

In [48]:
local_dataset = pd.read_csv("data/local/cebu.csv")
local_dataset.head()

,Patient,Age,Height (cm),Weight (kg),Body Mass Index (BMI),Blood Pressure,Menstrual Irregularities,Symptoms,Ultrasound Findings,Gravida
0,P1,34,1.57,64.0,25.9645,120/80,Irregular,"irregular masses, weight gain, hirsutism",PCO,G0P0
1,P2,22,1.53,52.0,22.2137,120/80,Irregular,"irregular masses, acne, weight gain",PCO,G0P0
2,P3,19,1.55,55.0,22.8928,110/70,Regular,"irregular masses, acne",PCO,G0P0
3,P4,24,1.55,55.0,22.8928,110/70,Irregular,"irregular masses, acne, weight gain",PCO,G0P0
4,P5,25,1.59,50.0,19.7777,110/71,Regular,irregular masses,PCO,G0P0


In [49]:
local_dataset.iloc[49]

Patient                         P50
Age                              25
Height (cm)                    1.56
Weight (kg)                    64.0
Body Mass Index (BMI)          26.3
Blood Pressure               110/80
Menstrual Irregularities    Regular
Symptoms                        NaN
Ultrasound Findings          Normal
Gravida                        G0P0
Name: 49, dtype: object

In [50]:
global_dataset = pd.read_csv("data/global/dataset_1.csv")
global_dataset.head()

,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Pimples(Y/N),Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm)
0,1,1,0,28,44.6,152.0,19.300000,15,78,22,...,0,1.0,0,110,80,3,3,18.0,18.0,8.5
1,2,2,0,36,65.0,161.5,24.921163,15,74,20,...,0,0.0,0,120,70,3,5,15.0,14.0,3.7
2,3,3,1,33,68.8,165.0,25.270891,11,72,18,...,1,1.0,0,120,80,13,15,18.0,20.0,10.0
3,4,4,0,37,65.0,148.0,29.674945,13,72,20,...,0,0.0,0,120,70,2,2,15.0,14.0,7.5
4,5,5,0,25,52.0,161.0,20.060954,11,72,18,...,0,0.0,0,120,80,3,4,16.0,14.0,7.0


## Preprocessing

In [51]:
from adapter import Dataset1Adapter, LocalDatasetAdapter

### Global

In [52]:
adapter = Dataset1Adapter(global_dataset)
preprocessed_global_dataset = adapter.convert()

/Users/nickanthonymiras/Programming/projects/pcos-analysis/adapter.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataframe[features.menstrual_irregularity] = filtered_dataframe[features.menstrual_irregularity].map({2 : 0, 4 : 1, 5 : 1})


In [53]:
# MinMaxScaler
preprocessed_global_dataset['bmi'] = MinMaxScaler().fit_transform(preprocessed_global_dataset[['bmi']])
preprocessed_global_dataset['age'] = MinMaxScaler().fit_transform(preprocessed_global_dataset[['age']])

In [54]:
preprocessed_global_dataset.head()

,bmi,age,has_pcos,hirsutism,acne,infertility,menstrual_irregularity
0,0.259878,0.285714,0,0,0,0,0
1,0.472141,0.571429,0,0,0,1,0
2,0.485347,0.464286,1,0,1,1,0
3,0.651650,0.607143,0,0,0,0,0
4,0.288613,0.178571,0,0,0,1,0


In [55]:
preprocessed_global_dataset.dtypes

bmi                       float64
age                       float64
has_pcos                    int64
hirsutism                   int64
acne                        int64
infertility                 int64
menstrual_irregularity      int64
dtype: object

### Local

Symptom Types

In [56]:
local_dataset['Symptoms'].dropna().str.lower().str.split(", ").explode().unique()

array(['irregular masses', 'weight gain', 'hirsutism', 'acne',
       'myofascial pain', 'diabetis', 'vaginal discharge', 'amenorrhea',
       'heavy menstrual bleeding', 'hair loss', 'pimples', 'hair growth',
       'skin darkening'], dtype=object)

In [57]:
adapter = LocalDatasetAdapter(local_dataset)
preprocessed_local_dataset = adapter.convert()

/Users/nickanthonymiras/Programming/projects/pcos-analysis/adapter.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataframe[features.infertility] = filtered_dataframe[features.infertility].apply(lambda x: 1 if x == 'G1P0' else 0)
/Users/nickanthonymiras/Programming/projects/pcos-analysis/adapter.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataframe[features.menstrual_irregularity] = filtered_dataframe[features.menstrual_irregularity].apply(lambda x: 1 if x == "Irregular" els

In [58]:
# MinMaxScaler
preprocessed_local_dataset['bmi'] = MinMaxScaler().fit_transform(preprocessed_local_dataset[['bmi']])
preprocessed_local_dataset['age'] = MinMaxScaler().fit_transform(preprocessed_local_dataset[['age']])

In [59]:
preprocessed_local_dataset.head()

,bmi,age,has_pcos,infertility,menstrual_irregularity,acne,hirsutism
0,0.456999,0.625000,1,0,1,0,1
1,0.320574,0.125000,1,0,1,1,0
2,0.345274,0.000000,1,0,0,1,0
3,0.345274,0.208333,1,0,1,1,0
4,0.231971,0.250000,1,0,0,0,0


## Quick Modelling and Analysis

In [60]:
# Define features and target variable
X = preprocessed_local_dataset.drop(columns=['has_pcos'])
y = preprocessed_local_dataset['has_pcos']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Bernoulli Naive Bayes
The Bernoulli-type of the Naive Bayes works really well for binary features.

In [61]:
bnb = BernoulliNB(alpha=1.0)

# 4. Train the model
bnb.fit(X_train, y_train)

# 5. Make predictions
y_pred = bnb.predict(X_test)

# 6. Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.85

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.85      1.00      0.92        17

    accuracy                           0.85        20
   macro avg       0.42      0.50      0.46        20
weighted avg       0.72      0.85      0.78        20



/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nickanthonymiras/miniconda3/envs/datascience/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

### Random Forest

In [62]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))    

Confusion Matrix:
[[ 1  2]
 [ 2 15]]

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.88      0.88      0.88        17

    accuracy                           0.80        20
   macro avg       0.61      0.61      0.61        20
weighted avg       0.80      0.80      0.80        20



### XGBoost

In [63]:
model = xgb.XGBClassifier(
    n_estimators=100,      # Number of gradient boosted trees
    learning_rate=0.1,     # Step size shrinkage used to prevent overfitting
    max_depth=3,           # Maximum depth of a tree
    random_state=42,       # Seed for reproducibility
    eval_metric='logloss'  # Evaluation metric for validation data
)

print("Training the XGBoost model...")
model.fit(X_train, y_train)

print("Making predictions...")
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Training the XGBoost model...
Making predictions...

Model Accuracy: 80.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.88      0.88      0.88        17

    accuracy                           0.80        20
   macro avg       0.61      0.61      0.61        20
weighted avg       0.80      0.80      0.80        20



### KNN with Hamming

In [64]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report

knn_hamming = KNeighborsClassifier(n_neighbors=5, metric='hamming')
knn_hamming.fit(X_train, y_train)

y_pred_knn = knn_hamming.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

Confusion Matrix:
[[ 1  2]
 [ 2 15]]

Classification Report:
              precision    recall  f1-score   support

           0       0.33      0.33      0.33         3
           1       0.88      0.88      0.88        17

    accuracy                           0.80        20
   macro avg       0.61      0.61      0.61        20
weighted avg       0.80      0.80      0.80        20



### Dataset Augmentation with Global Dataset

In [65]:
import importlib
import adapter
importlib.reload(adapter)
from adapter import Dataset1Adapter, LocalDatasetAdapter

adapter_global = Dataset1Adapter(global_dataset)
preprocessed_global_dataset = adapter_global.convert()

# Combine datasets
# Ensure columns are in same order
columns = preprocessed_local_dataset.columns
preprocessed_global_dataset = preprocessed_global_dataset[columns]

# Concatenate
augmented_dataset = pd.concat([preprocessed_local_dataset, preprocessed_global_dataset], axis=0).reset_index(drop=True)

print(f"Local dataset shape: {preprocessed_local_dataset.shape}")
print(f"Global dataset shape: {preprocessed_global_dataset.shape}")
print(f"Augmented dataset shape: {augmented_dataset.shape}")

augmented_dataset.head()

Local dataset shape: (100, 7)
Global dataset shape: (541, 7)
Augmented dataset shape: (641, 7)


/Users/nickanthonymiras/Programming/projects/pcos-analysis/adapter.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_dataframe[features.menstrual_irregularity] = filtered_dataframe[features.menstrual_irregularity].map({2 : 0, 4 : 1, 5 : 1})


,bmi,age,has_pcos,infertility,menstrual_irregularity,acne,hirsutism
0,0.456999,0.625000,1,0,1,0,1
1,0.320574,0.125000,1,0,1,1,0
2,0.345274,0.000000,1,0,0,1,0
3,0.345274,0.208333,1,0,1,1,0
4,0.231971,0.250000,1,0,0,0,0


### Modeling with Augmented Data
Train an XGBoost model on the combined dataset to evaluate performance improvements.

In [66]:
# Define features and target variable
X = augmented_dataset.drop(columns=['has_pcos', 'bmi', 'age'])
y = augmented_dataset['has_pcos']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [67]:

model = xgb.XGBClassifier(
    n_estimators=100,      # Number of gradient boosted trees
    learning_rate=0.1,     # Step size shrinkage used to prevent overfitting
    max_depth=3,           # Maximum depth of a tree
    random_state=42,       # Seed for reproducibility
    eval_metric='logloss'  # Evaluation metric for validation data
)

print("Training the XGBoost model on augmented dataset...")
model.fit(X_train, y_train)

print("Making predictions...")
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

Training the XGBoost model on augmented dataset...
Making predictions...

Model Accuracy: 75.97%

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.86      0.82        83
           1       0.69      0.59      0.64        46

    accuracy                           0.76       129
   macro avg       0.74      0.72      0.73       129
weighted avg       0.75      0.76      0.75       129



In [68]:
knn_hamming = KNeighborsClassifier(n_neighbors=5, metric='hamming')
knn_hamming.fit(X_train, y_train)

y_pred_knn = knn_hamming.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

Confusion Matrix:
[[75  8]
 [20 26]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.90      0.84        83
           1       0.76      0.57      0.65        46

    accuracy                           0.78       129
   macro avg       0.78      0.73      0.75       129
weighted avg       0.78      0.78      0.77       129



### Bernoulli Naive Bayes

In [69]:
bnb = BernoulliNB(alpha=1.0)

# 4. Train the model
bnb.fit(X_train, y_train)

# 5. Make predictions
y_pred = bnb.predict(X_test)

# 6. Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7286821705426356

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.77      0.79        83
           1       0.61      0.65      0.63        46

    accuracy                           0.73       129
   macro avg       0.71      0.71      0.71       129
weighted avg       0.73      0.73      0.73       129



### Random Forest

In [70]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))    

Confusion Matrix:
[[71 12]
 [19 27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.86      0.82        83
           1       0.69      0.59      0.64        46

    accuracy                           0.76       129
   macro avg       0.74      0.72      0.73       129
weighted avg       0.75      0.76      0.75       129



The `Dataset1Adapter` has been updated to correctly map `Cycle(R/I)` values and standardize column names, ensuring compatibility with the local dataset for augmentation.
The augmented dataset combines 541 global samples + 90 local samples (approx) for a total of ~630 samples.
Training on the augmented dataset should yield better generalization and potentially higher accuracy.

# Conclusion